# Google Calendar MCP Client Demo

This notebook demonstrates how to authenticate via the full OAuth dance and use the refactored `CalendarClient` directly. It coordinates with `MeetClient` and `DriveClient` internally.

In [ ]:
import os
import sys
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

sys.path.append('..')
from mcp_servers.google_calendar.app.connector import EventsClient


SCOPES = [
    'https://www.googleapis.com/auth/meetings.space.readonly',
    'https://www.googleapis.com/auth/calendar.events.readonly',
]

def get_credentials():
    creds = None
    if os.path.exists('token.json'):
        try:
            creds = Credentials.from_authorized_user_file('token.json', SCOPES)
        except ValueError:
            creds = None
            os.remove('token.json')
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('client_secret.json', SCOPES)
            creds = flow.run_local_server(port=51063)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

creds = get_credentials()
events_client = EventsClient(creds)

In [ ]:
calendar_events = events_client.list_events(max_events=100, date_min = "2026-03-01", date_max="2026-04-01")

In [ ]:
calendar_events

In [ ]:
calendar_event = calendar_events[4]
calendar_event

In [ ]:
calendar_event.attendees

In [ ]:
meeting_code = calendar_event.meet_session.meeting_code
meeting_code

In [ ]:
meet_sessions = events_client.list_meet_sessions(meeting_code=meeting_code)
meet_sessions

In [ ]:
meet_session = meet_sessions[0]
meet_session

In [ ]:
meet_session_id = meet_session.session_id
meet_session_id

In [ ]:
events_client.list_meet_participants(meet_session_id=meet_session_id)

In [ ]:
recording_id = meet_session.recordings.recording_ids[0]
recording_id

In [ ]:
events_client.get_meet_recording(recording_id)

In [ ]:
events_client.list_meet_participants(meet_session_id = meet_session_id)

In [ ]:
meet_session_id